# 📘 Notebook 12: Recursion

### Course: *From Zero to Data Structures & Algorithms in Python*
**Instructor:** Ioannis Tsioulis

*Module C: Algorithms & Complexity · Notebook 2 of 4 in this module · (12 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain what a **recursive function** is
- Identify the two parts every recursion needs: the **base case** and the **recursive case**
- Trace a recursive call by hand and describe the **call stack**
- Rewrite simple loops as recursion, and recursion as loops
- Recognise when recursion is expensive, and repair it with **memoisation**
- Apply recursion to problems that are naturally nested

> **Prerequisites:** Notebooks 04 (functions) and 11 (Big O).
>
> 🔭 **Why this notebook matters later:** merge sort (Notebook 14) and every operation on trees (Notebook 18) are written recursively. Some problems are awkward with loops and almost effortless with recursion. This notebook gives you the tool before you need it.

## 1. A function that calls itself

### Intuition first
You are standing in a long queue and want to know your position. You cannot see the front. So you ask the person ahead of you: *"what is your position?"* They do not know either, so they ask the person ahead of them, and so on. Eventually the question reaches the first person, who answers *"I am number 1."* The answer then travels back: 2, 3, 4, until it reaches you.

Nobody solved the whole problem. Each person solved a tiny part (*"my position is one more than the position ahead of me"*) and passed a **smaller version of the same problem** along.

### Formal definition
A **recursive function** is a function that calls itself. Every correct recursion has exactly two ingredients:

1. A **base case**: an input so simple that the answer is known immediately, with no further calls. (The first person in the queue.)
2. A **recursive case**: the function calls itself on a **smaller** input, and uses that result to build its own answer.

### The standard example: the factorial
The factorial of `n`, written `n!`, is the product of all whole numbers from 1 to `n`. So `5! = 5 × 4 × 3 × 2 × 1 = 120`.

Look at it differently: `5! = 5 × 4!`. The factorial of 5 is defined **in terms of a smaller factorial**. That is a recursive definition, and it translates into code almost word for word.

In [ ]:
def factorial(n):
    if n == 1:                         # base case: we know the answer
        return 1
    return n * factorial(n - 1)        # recursive case: a smaller problem

print(factorial(5))
print(factorial(10))

## 2. What actually happens: the call stack

It is worth following one call slowly. To compute `factorial(4)`, Python must first compute `factorial(3)`, which needs `factorial(2)`, which needs `factorial(1)`.

Each unfinished call **waits** for the one below it. Python keeps these waiting calls in a pile called the **call stack**. When the base case is reached, the calls finish one by one, in reverse order.

```
factorial(4)
= 4 * factorial(3)
= 4 * (3 * factorial(2))
= 4 * (3 * (2 * factorial(1)))      <- base case reached
= 4 * (3 * (2 * 1))
= 4 * (3 * 2)
= 4 * 6
= 24
```

The version below prints a message on the way **down** and on the way back **up**. The indentation shows how deep we are.

In [ ]:
def factorial_traced(n, depth=0):
    indent = "    " * depth
    print(f"{indent}factorial({n}) called")
    if n == 1:
        print(f"{indent}base case reached, returning 1")
        return 1
    result = n * factorial_traced(n - 1, depth + 1)
    print(f"{indent}factorial({n}) returns {result}")
    return result

factorial_traced(4)

> ⚠️ **Common pitfalls**
>
> - **No base case**, or a base case that is never reached. The function then calls itself forever, until Python stops it with a `RecursionError`. Calling `factorial(0)` above does exactly that: the argument goes 0, -1, -2, ... and never equals 1.
> - A recursive case that does not make the problem **smaller**. Every call must move towards the base case.
> - Forgetting `return` in front of the recursive call. The function then computes the answer and throws it away.

## 3. Thinking recursively

To write a recursive function, do not try to follow all the calls in your head. Answer just two questions:

1. **What is the simplest possible input, and what is its answer?** That is the base case.
2. **If someone handed me the answer for a slightly smaller input, how would I finish the job?** That is the recursive case.

Then trust the function to do its work on the smaller input.

### A worked example: the sum of a list
- Simplest input: an empty list. Its sum is 0.
- Smaller input: the list without its first element. The sum of the whole list is the first element **plus** the sum of the rest.

In [ ]:
def sum_list(numbers):
    if len(numbers) == 0:                       # base case
        return 0
    return numbers[0] + sum_list(numbers[1:])   # first element + sum of the rest

print(sum_list([3, 7, 2, 8]))

### A worked example: counting down
Not every recursive function returns a value. Here the work is done by printing. Notice that the order of the two lines in the recursive case decides whether we count down or up.

In [ ]:
def countdown(n):
    if n == 0:
        print("Liftoff!")
        return
    print(n)               # print BEFORE the recursive call: 3, 2, 1
    countdown(n - 1)

def countup(n):
    if n == 0:
        return
    countup(n - 1)
    print(n)               # print AFTER the recursive call: 1, 2, 3

countdown(3)
print("---")
countup(3)

### A worked example: reversing a string
- Simplest input: an empty string, or a single character. It is its own reverse.
- Smaller input: the string without its first character. Reverse that, then put the first character **at the end**.

In [ ]:
def reverse(text):
    if len(text) <= 1:
        return text
    return reverse(text[1:]) + text[0]

print(reverse("Python"))

## 4. Recursion or loop?

Everything above can also be written with a loop. Here are both versions of the factorial.

In [ ]:
def factorial_loop(n):
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result

print(factorial_loop(5), factorial(5))

Both are O(n): the loop runs `n` times, and the recursion makes `n` calls. So which should we use?

| | Loop | Recursion |
|---|---|---|
| Speed | slightly faster | a little slower, each call has a small cost |
| Memory | constant | one waiting call per level |
| Depth | unlimited | Python allows about 1,000 nested calls by default |
| Best for | simple, flat repetition | problems that are **nested** or **split into parts** |

For a simple count, use a loop. Recursion earns its place when the problem itself is recursive: a folder that contains folders, a list that contains lists, a problem that splits into two halves. The exercises and the next notebooks contain several such problems.

## 5. When recursion becomes expensive: Fibonacci

### The sequence
In the **Fibonacci sequence**, each number is the sum of the two before it: 0, 1, 1, 2, 3, 5, 8, 13, 21, ... The definition is recursive, with **two** base cases:

- `fib(0) = 0` and `fib(1) = 1`
- `fib(n) = fib(n - 1) + fib(n - 2)`

In [ ]:
def fib(n):
    if n <= 1:
        return n
    return fib(n - 1) + fib(n - 2)

for i in range(10):
    print(fib(i), end=" ")

The code is beautiful, and it hides a serious problem. Let us time it as `n` grows.

In [ ]:
import time

for n in [10, 20, 25, 28]:
    start = time.perf_counter()
    result = fib(n)
    elapsed = time.perf_counter() - start
    print(f"fib({n}) = {result:>7}   {elapsed:.4f} seconds")

### Why it is slow
Each call makes **two** further calls, each of those makes two more, and so on. Worse, the same values are computed again and again: to compute `fib(5)`, the function computes `fib(3)` twice and `fib(2)` three times.

```
                    fib(5)
                 /          \
            fib(4)            fib(3)
           /      \           /     \
       fib(3)    fib(2)    fib(2)   fib(1)
       /    \     /   \     /   \
   fib(2) fib(1) ...  ...  ...  ...
```

Let us count the calls. The number grows by more than half every time `n` increases by one. This is **exponential** growth, written O(2ⁿ), far worse than the O(n²) of the previous notebook.

In [ ]:
call_count = 0

def fib_counted(n):
    global call_count
    call_count += 1
    if n <= 1:
        return n
    return fib_counted(n - 1) + fib_counted(n - 2)

for n in [5, 10, 20, 25]:
    call_count = 0
    fib_counted(n)
    print(f"fib({n}) needed {call_count:>7,} calls")

### The repair: memoisation
The cure is simple: **remember every answer the first time it is computed**. Before computing anything, check whether we already have it. The memory is a dictionary, whose lookups are O(1). This technique is called **memoisation**.

In [ ]:
memory = {}

def fib_memo(n):
    if n in memory:              # already computed: answer instantly
        return memory[n]
    if n <= 1:
        return n
    result = fib_memo(n - 1) + fib_memo(n - 2)
    memory[n] = result           # remember it for next time
    return result

start = time.perf_counter()
print(fib_memo(80))
print(f"{time.perf_counter() - start:.6f} seconds")

With the plain version, `fib(80)` would need about 76,000 million million calls. Even at a hundred million calls per second, that is more than twenty years. With memoisation each value is computed **once**, so the cost falls from O(2ⁿ) to O(n).

> 🧠 The lesson reaches far beyond Fibonacci. A dramatic speed-up came not from a faster computer, but from a better idea combined with the right data structure. That is the theme of this whole course.

## 6. Where recursion shines: nested data

Here is a problem that is awkward with loops. A list contains numbers and other lists, which may contain further lists, to any depth. We want the sum of all the numbers.

A loop would need to know the depth in advance. Recursion does not care: *if the item is a number, add it; if it is a list, the sum of that list is the same problem again.*

In [ ]:
def nested_sum(items):
    total = 0
    for item in items:
        if isinstance(item, list):       # is this item itself a list?
            total += nested_sum(item)    # the same problem, one level deeper
        else:
            total += item
    return total

print(nested_sum([1, [2, 3], [4, [5, [6, 7]]], 8]))

The built-in `isinstance(value, type)` asks whether a value is of a given type. Notice the shape of this function: a loop over the items, and a recursive call for the items that are themselves containers. We will see exactly this shape again with trees.

---
## ✏️ Exercises

For each exercise, write down the base case and the recursive case before you write any code.

### Exercise 1 (Sum of the first n numbers)
Write a recursive function `sum_to(n)` that returns `1 + 2 + ... + n`.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def sum_to(n):
    if n == 0:
        return 0
    return n + sum_to(n - 1)

print(sum_to(5))
print(sum_to(100))
```
</details>

### Exercise 2 (Power)
Write a recursive function `power(base, exponent)` that returns `base` raised to a non-negative whole `exponent`, without using `**`. (Hint: anything to the power 0 is 1.)

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def power(base, exponent):
    if exponent == 0:
        return 1
    return base * power(base, exponent - 1)

print(power(2, 10))
print(power(5, 3))
```
</details>

### Exercise 3 (Sum of the digits)
Write a recursive function `digit_sum(n)` that returns the sum of the digits of a non-negative whole number. For example `digit_sum(4721)` is `14`. (Hint: `n % 10` is the last digit, and `n // 10` is the number without it.)

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def digit_sum(n):
    if n < 10:
        return n
    return n % 10 + digit_sum(n // 10)

print(digit_sum(4721))
```
</details>

### Exercise 4 (Count the occurrences)
Write a recursive function `count_value(numbers, target)` that returns how many times `target` appears in the list.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def count_value(numbers, target):
    if len(numbers) == 0:
        return 0
    if numbers[0] == target:
        return 1 + count_value(numbers[1:], target)
    return count_value(numbers[1:], target)

print(count_value([3, 1, 3, 7, 3], 3))
```
</details>

### Exercise 5 (Recursive palindrome)
Write a recursive function `is_palindrome(text)`. (Hint: a text is a palindrome if its first and last characters are equal **and** the part between them is a palindrome.)

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def is_palindrome(text):
    if len(text) <= 1:
        return True
    if text[0] != text[-1]:
        return False
    return is_palindrome(text[1:-1])

print(is_palindrome("level"))
print(is_palindrome("python"))
```
</details>

### Exercise 6 (Greatest common divisor)
The algorithm of Euclid, more than 2,000 years old, finds the greatest common divisor of two numbers: `gcd(a, b)` is `a` when `b` is 0, and otherwise it equals `gcd(b, a % b)`. Write it as a recursive function and test it with `gcd(48, 18)`.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def gcd(a, b):
    if b == 0:
        return a
    return gcd(b, a % b)

print(gcd(48, 18))
print(gcd(17, 5))
```
</details>

### Exercise 7 (Flatten a nested list)
Write a recursive function `flatten(items)` that turns a nested list such as `[1, [2, 3], [4, [5, 6]]]` into the flat list `[1, 2, 3, 4, 5, 6]`.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def flatten(items):
    result = []
    for item in items:
        if isinstance(item, list):
            result = result + flatten(item)
        else:
            result.append(item)
    return result

print(flatten([1, [2, 3], [4, [5, 6]]]))
```
</details>

### Exercise 8 (The Towers of Hanoi, a little harder)
Three pegs, `A`, `B` and `C`. A tower of `n` discs of decreasing size sits on peg `A`. Move the whole tower to peg `C`, moving one disc at a time and never placing a larger disc on a smaller one. Write `hanoi(n, source, target, spare)` that prints every move. (Hint: to move `n` discs, first move the top `n - 1` discs to the spare peg, then move the largest disc, then move the `n - 1` discs on top of it.)

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def hanoi(n, source, target, spare):
    if n == 0:
        return
    hanoi(n - 1, source, spare, target)             # clear the way
    print(f"Move disc {n} from {source} to {target}")
    hanoi(n - 1, spare, target, source)             # put the rest back on top

hanoi(3, "A", "C", "B")
```

*Three discs need 7 moves, and `n` discs need `2ⁿ - 1`. The solution is three lines long, yet very hard to write with loops. This is the kind of problem recursion was made for.*
</details>

## 🔑 Key takeaways

- A **recursive function** calls itself on a **smaller** version of the same problem.
- Every recursion needs a **base case** (answered directly) and a **recursive case** (moves towards the base case).
- Unfinished calls wait on the **call stack** and complete in reverse order.
- To write one, ask: what is the simplest input, and how do I finish the job given the answer to a smaller input?
- Loops suit flat repetition; recursion suits problems that are **nested** or that **split into parts**.
- Naive recursion can repeat work exponentially; **memoisation** (remembering answers in a dictionary) repairs it.

---
**Next:** *Notebook 13: Searching*, where halving a problem turns a million steps into twenty.

---
*© Ioannis Tsioulis. *From Zero to Data Structures & Algorithms in Python*. Prepared for educational use.*